In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    brier_score_loss, log_loss, classification_report, accuracy_score
)

# Create the plots directory if it doesn't exist
os.makedirs("plots", exist_ok=True)

pd.set_option("display.width", 120)

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


print("1) CALIBRATION")


# Gradient Boosting tends to be overconfident — good candidate to demonstrate calibration
gb_model = GradientBoostingClassifier(n_estimators=300, random_state=42)
gb_model.fit(X_train, y_train)
y_scores_uncalibrated = gb_model.predict_proba(X_test)[:, 1]

brier_before = brier_score_loss(y_test, y_scores_uncalibrated)
print(f"Brier score BEFORE calibration: {brier_before:.4f}  (lower is better)")

calibrated_gb = CalibratedClassifierCV(gb_model, method="isotonic", cv=5)
calibrated_gb.fit(X_train, y_train)
y_scores_calibrated = calibrated_gb.predict_proba(X_test)[:, 1]

brier_after = brier_score_loss(y_test, y_scores_calibrated)
print(f"Brier score AFTER calibration : {brier_after:.4f}")

# Reliability diagram
prob_true_before, prob_pred_before = calibration_curve(y_test, y_scores_uncalibrated, n_bins=10, strategy="quantile")
prob_true_after, prob_pred_after = calibration_curve(y_test, y_scores_calibrated, n_bins=10, strategy="quantile")

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
plt.plot(prob_pred_before, prob_true_before, "o-", label="Before calibration")
plt.plot(prob_pred_after, prob_true_after, "o-", label="After calibration (isotonic)")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.title("Reliability Diagram")
plt.legend()
plt.savefig("plots/26_calibration.png", bbox_inches="tight")
plt.close()
print("Saved reliability diagram to plots/26_calibration.png")


print("2) PROBABILITY PREDICTION — comparing native probability quality across models")


candidates = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=300, random_state=42),
    "SVM (probability=True)": SVC(probability=True, random_state=42),
}

print(f"{'Model':28s} {'Accuracy':>10s} {'Log Loss':>10s} {'Brier':>10s}")
for name, clf in candidates.items():
    clf.fit(X_train, y_train)
    probs = clf.predict_proba(X_test)[:, 1]
    preds = clf.predict(X_test)
    print(f"{name:28s} {accuracy_score(y_test, preds):10.4f} "
          f"{log_loss(y_test, probs):10.4f} {brier_score_loss(y_test, probs):10.4f}")

print("\n(Two models can have similar accuracy but very different log loss / Brier score —")
print(" that gap reflects how trustworthy their probabilities are, not just their final labels.)")


print("3) VOTING CLASSIFIERS (hard vs soft)")


clf1 = LogisticRegression(max_iter=5000)
clf2 = RandomForestClassifier(n_estimators=200, random_state=42)
clf3 = SVC(probability=True, random_state=42)

hard_voting = VotingClassifier(estimators=[("lr", clf1), ("rf", clf2), ("svc", clf3)], voting="hard")
soft_voting = VotingClassifier(estimators=[("lr", clf1), ("rf", clf2), ("svc", clf3)], voting="soft")

hard_voting.fit(X_train, y_train)
soft_voting.fit(X_train, y_train)

print(f"Hard voting test accuracy: {hard_voting.score(X_test, y_test):.4f}")
print(f"Soft voting test accuracy: {soft_voting.score(X_test, y_test):.4f}")
print("(Soft voting uses predicted probabilities, hard voting uses only the majority class label)")


print("4) ENSEMBLE STACKING")


base_estimators = [
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42)),
    ("gb", GradientBoostingClassifier(n_estimators=200, random_state=42)),
    ("svc", SVC(probability=True, random_state=42)),
]

stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=5000),
    cv=5,  # generates out-of-fold predictions internally to avoid leakage into the meta-model
    stack_method="predict_proba"
)
stacking_model.fit(X_train, y_train)
print(f"Stacking test accuracy: {stacking_model.score(X_test, y_test):.4f}")

# Compare individual base models vs. the stack
print("\nIndividual base model accuracies (for comparison):")
for name, est in base_estimators:
    est_clone = est.__class__(**est.get_params())
    est_clone.fit(X_train, y_train)
    print(f"  {name}: {est_clone.score(X_test, y_test):.4f}")
print(f"  STACK: {stacking_model.score(X_test, y_test):.4f}")


print("5) BLENDING (manual — single holdout split instead of k-fold)")


# Split the training set into a base-train portion and a holdout portion
X_base, X_holdout, y_base, y_holdout = train_test_split(
    X_train, y_train, test_size=0.3, random_state=42, stratify=y_train
)

rf_blend = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_base, y_base)
gb_blend = GradientBoostingClassifier(n_estimators=200, random_state=42).fit(X_base, y_base)

# Meta-features = base models' predictions on the HOLDOUT set only
holdout_meta_features = pd.DataFrame({
    "rf": rf_blend.predict_proba(X_holdout)[:, 1],
    "gb": gb_blend.predict_proba(X_holdout)[:, 1],
})
meta_model_blend = LogisticRegression().fit(holdout_meta_features, y_holdout)

# At inference time: generate the same meta-features on the test set
test_meta_features = pd.DataFrame({
    "rf": rf_blend.predict_proba(X_test)[:, 1],
    "gb": gb_blend.predict_proba(X_test)[:, 1],
})
blend_preds = meta_model_blend.predict(test_meta_features)
print(f"Blending test accuracy: {accuracy_score(y_test, blend_preds):.4f}")
print("(Blending trades some data efficiency for speed — base models never see the holdout portion)")


print("6) MODEL UNCERTAINTY — variance across a Random Forest's individual trees")


rf_uncertainty = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_train, y_train)

tree_preds = np.array([tree.predict_proba(X_test.values)[:, 1] for tree in rf_uncertainty.estimators_])
mean_pred = tree_preds.mean(axis=0)
uncertainty = tree_preds.std(axis=0)  # high std = trees disagree = high uncertainty

uncertainty_df = pd.DataFrame({
    "mean_prediction": mean_pred,
    "uncertainty_std": uncertainty,
    "true_label": y_test
}).sort_values("uncertainty_std", ascending=False)

print("Top 10 MOST uncertain test predictions (trees disagree most):")
print(uncertainty_df.head(10))
print("\nTop 10 LEAST uncertain test predictions (trees agree most):")
print(uncertainty_df.tail(10))

# Sanity check: are more uncertain predictions more often wrong?
final_preds = rf_uncertainty.predict(X_test)
is_correct = (final_preds == y_test)
print(f"\nMean uncertainty when prediction is CORRECT : {uncertainty[is_correct].mean():.4f}")
print(f"Mean uncertainty when prediction is WRONG   : {uncertainty[~is_correct].mean():.4f}")
print("(We'd expect uncertainty to be higher on wrong predictions, if the signal is meaningful)")


print("7) ERROR ANALYSIS")


final_model = stacking_model
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

errors_mask = y_pred != y_test
errors_df = X_test[errors_mask].copy()
errors_df["true_label"] = y_test[errors_mask]
errors_df["predicted_label"] = y_pred[errors_mask]
errors_df["predicted_prob"] = y_prob[errors_mask]
errors_df["confidence_in_wrong_answer"] = np.abs(errors_df["predicted_prob"] - 0.5) * 2  # 0=unsure, 1=fully confident

print(f"Total test errors: {errors_mask.sum()} out of {len(y_test)}")
print("\nAll misclassified instances (sorted by confidence in the WRONG answer):")
print(errors_df[["true_label", "predicted_label", "predicted_prob", "confidence_in_wrong_answer"]]
      .sort_values("confidence_in_wrong_answer", ascending=False))

if errors_mask.sum() > 0:
    high_conf_errors = errors_df[errors_df["confidence_in_wrong_answer"] > 0.5]
    print(f"\nHigh-confidence errors (model was quite sure but wrong): {len(high_conf_errors)}")
    print("These are the most informative to investigate — the model isn't just 'unsure',")
    print("it's confidently relying on a pattern that misled it for these specific cases.")

print("\nFull classification report on the final stacked model:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

print("\nDone. Plots saved under plots/")

1) CALIBRATION
Brier score BEFORE calibration: 0.0423  (lower is better)
Brier score AFTER calibration : 0.0293
Saved reliability diagram to plots/26_calibration.png
2) PROBABILITY PREDICTION — comparing native probability quality across models
Model                          Accuracy   Log Loss      Brier
Logistic Regression              0.9649     0.0894     0.0285
Random Forest                    0.9474     0.1094     0.0321
Gradient Boosting                0.9561     0.2123     0.0423
SVM (probability=True)           0.9298     0.2016     0.0605

(Two models can have similar accuracy but very different log loss / Brier score —
 that gap reflects how trustworthy their probabilities are, not just their final labels.)
3) VOTING CLASSIFIERS (hard vs soft)
Hard voting test accuracy: 0.9561
Soft voting test accuracy: 0.9474
(Soft voting uses predicted probabilities, hard voting uses only the majority class label)
4) ENSEMBLE STACKING
Stacking test accuracy: 0.9561

Individual base model a